# 장기 가격 데이터 수집 노트북

`yfinance`를 사용해 가능한 한 긴 가격 이력을 내려받아 `long_price_master_history.csv`로 저장합니다.

## Overview

- Input universe: curated US stock/ETF universe (~100 tickers)
- Download window: `period="max"`
- Retention window: keep up to the most recent 50 years only
- Output file: `long_price_master_history.csv`
- Downstream notebook: `4_price_walkforward_analysis.ipynb`


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("yfinance가 설치되어 있어야 합니다. `pip install yfinance` 후 다시 실행해주세요.") from exc

In [2]:
WORKSPACE_ROOT = Path.cwd()
if (WORKSPACE_ROOT / "1_data_collection.ipynb").exists() or (WORKSPACE_ROOT / "3_advanced_analysis.ipynb").exists():
    STOCK_ANALYSIS_DIR = WORKSPACE_ROOT
elif (WORKSPACE_ROOT / "Stock_analysis" / "1_data_collection.ipynb").exists():
    STOCK_ANALYSIS_DIR = WORKSPACE_ROOT / "Stock_analysis"
else:
    STOCK_ANALYSIS_DIR = WORKSPACE_ROOT / "Stock_analysis"

DATA_DIR = STOCK_ANALYSIS_DIR / "data" / "event_driven_stock_predictor"
PRICE_MASTER_PATH = DATA_DIR / "price_master_history.csv"
LONG_PRICE_MASTER_PATH = DATA_DIR / "long_price_master_history.csv"
LONG_LOOKBACK_PERIOD = "max"
MAX_HISTORY_YEARS = 30
HISTORY_START_CUTOFF = (pd.Timestamp.utcnow().floor("D") - pd.DateOffset(years=MAX_HISTORY_YEARS))

print(f"Stock analysis dir: {STOCK_ANALYSIS_DIR}")
print(f"Base price path: {PRICE_MASTER_PATH}")
print(f"Long price path: {LONG_PRICE_MASTER_PATH}")
print(f"History cutoff: {HISTORY_START_CUTOFF.date()} (keep up to {MAX_HISTORY_YEARS} years)")


Stock analysis dir: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis
Base price path: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\price_master_history.csv
Long price path: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\long_price_master_history.csv
History cutoff: 1996-03-27 (keep up to 30 years)


## 종목 목록 준비

In [3]:
DEFAULT_UNIVERSE = [
    ("AAPL", "Apple", "MegaCap_Tech"),
    ("MSFT", "Microsoft", "MegaCap_Tech"),
    ("NVDA", "NVIDIA", "MegaCap_Tech"),
    ("AMZN", "Amazon", "MegaCap_Tech"),
    ("GOOGL", "Alphabet", "MegaCap_Tech"),
    ("META", "Meta Platforms", "MegaCap_Tech"),
    ("TSLA", "Tesla", "MegaCap_Growth"),
    ("BRK-B", "Berkshire Hathaway", "MegaCap_Defensive"),
    ("JPM", "JPMorgan Chase", "Financials"),
    ("V", "Visa", "Financials"),
    ("MA", "Mastercard", "Financials"),
    ("BAC", "Bank of America", "Financials"),
    ("WFC", "Wells Fargo", "Financials"),
    ("GS", "Goldman Sachs", "Financials"),
    ("MS", "Morgan Stanley", "Financials"),
    ("BLK", "BlackRock", "Financials"),
    ("SPY", "SPDR S&P 500 ETF", "Index_ETF"),
    ("QQQ", "Invesco QQQ", "Index_ETF"),
    ("DIA", "SPDR Dow Jones ETF", "Index_ETF"),
    ("IWM", "iShares Russell 2000 ETF", "Index_ETF"),
    ("VTI", "Vanguard Total Stock Market ETF", "Index_ETF"),
    ("XLK", "Technology Select Sector SPDR", "Sector_ETF"),
    ("XLF", "Financial Select Sector SPDR", "Sector_ETF"),
    ("XLE", "Energy Select Sector SPDR", "Sector_ETF"),
    ("XLV", "Health Care Select Sector SPDR", "Sector_ETF"),
    ("XLI", "Industrial Select Sector SPDR", "Sector_ETF"),
    ("XLY", "Consumer Discretionary Select Sector SPDR", "Sector_ETF"),
    ("XLP", "Consumer Staples Select Sector SPDR", "Sector_ETF"),
    ("XLB", "Materials Select Sector SPDR", "Sector_ETF"),
    ("XLU", "Utilities Select Sector SPDR", "Sector_ETF"),
    ("VNQ", "Vanguard Real Estate ETF", "Sector_ETF"),
    ("SMH", "VanEck Semiconductor ETF", "Theme_ETF"),
    ("SOXX", "iShares Semiconductor ETF", "Theme_ETF"),
    ("IGV", "iShares Software ETF", "Theme_ETF"),
    ("ARKK", "ARK Innovation ETF", "Theme_ETF"),
    ("XBI", "SPDR Biotech ETF", "Theme_ETF"),
    ("ITA", "iShares US Aerospace & Defense ETF", "Theme_ETF"),
    ("GLD", "SPDR Gold Shares", "Commodity_ETF"),
    ("SLV", "iShares Silver Trust", "Commodity_ETF"),
    ("USO", "United States Oil Fund", "Commodity_ETF"),
    ("UNG", "United States Natural Gas Fund", "Commodity_ETF"),
    ("TLT", "iShares 20+ Year Treasury Bond ETF", "Macro_ETF"),
    ("IEF", "iShares 7-10 Year Treasury Bond ETF", "Macro_ETF"),
    ("HYG", "iShares iBoxx High Yield Corporate Bond ETF", "Macro_ETF"),
    ("LQD", "iShares iBoxx Investment Grade Corporate Bond ETF", "Macro_ETF"),
    ("KO", "Coca-Cola", "Defensive_Consumer"),
    ("PEP", "PepsiCo", "Defensive_Consumer"),
    ("PG", "Procter & Gamble", "Defensive_Consumer"),
    ("WMT", "Walmart", "Defensive_Consumer"),
    ("COST", "Costco", "Defensive_Consumer"),
    ("HD", "Home Depot", "Consumer"),
    ("MCD", "McDonald's", "Consumer"),
    ("NKE", "Nike", "Consumer"),
    ("SBUX", "Starbucks", "Consumer"),
    ("DIS", "Disney", "Consumer"),
    ("NFLX", "Netflix", "Media"),
    ("CSCO", "Cisco", "Enterprise_Tech"),
    ("ORCL", "Oracle", "Enterprise_Tech"),
    ("IBM", "IBM", "Enterprise_Tech"),
    ("ADBE", "Adobe", "Software"),
    ("CRM", "Salesforce", "Software"),
    ("INTU", "Intuit", "Software"),
    ("AMD", "AMD", "Semiconductors"),
    ("INTC", "Intel", "Semiconductors"),
    ("AVGO", "Broadcom", "Semiconductors"),
    ("QCOM", "Qualcomm", "Semiconductors"),
    ("TXN", "Texas Instruments", "Semiconductors"),
    ("MU", "Micron", "Semiconductors"),
    ("AMAT", "Applied Materials", "Semiconductors"),
    ("LRCX", "Lam Research", "Semiconductors"),
    ("ASML", "ASML", "Semiconductors"),
    ("PANW", "Palo Alto Networks", "Cybersecurity"),
    ("CRWD", "CrowdStrike", "Cybersecurity"),
    ("PLTR", "Palantir", "AI_Software"),
    ("SNOW", "Snowflake", "AI_Software"),
    ("UBER", "Uber", "Platform"),
    ("ABNB", "Airbnb", "Platform"),
    ("PYPL", "PayPal", "Fintech"),
    ("CAT", "Caterpillar", "Industrials"),
    ("DE", "Deere", "Industrials"),
    ("GE", "GE Aerospace", "Industrials"),
    ("HON", "Honeywell", "Industrials"),
    ("BA", "Boeing", "Industrials"),
    ("UNP", "Union Pacific", "Industrials"),
    ("UPS", "UPS", "Industrials"),
    ("FDX", "FedEx", "Industrials"),
    ("COP", "ConocoPhillips", "Energy"),
    ("XOM", "Exxon Mobil", "Energy"),
    ("CVX", "Chevron", "Energy"),
    ("SLB", "Schlumberger", "Energy"),
    ("NEE", "NextEra Energy", "Utilities"),
    ("JNJ", "Johnson & Johnson", "Healthcare"),
    ("PFE", "Pfizer", "Healthcare"),
    ("MRK", "Merck", "Healthcare"),
    ("LLY", "Eli Lilly", "Healthcare"),
    ("UNH", "UnitedHealth", "Healthcare"),
    ("ABBV", "AbbVie", "Healthcare"),
    ("TMO", "Thermo Fisher", "Healthcare"),
    ("ISRG", "Intuitive Surgical", "Healthcare"),
    ("DHR", "Danaher", "Healthcare"),
]


def load_symbol_universe() -> pd.DataFrame:
    universe = pd.DataFrame(DEFAULT_UNIVERSE, columns=["symbol", "asset_name", "category"])
    universe = universe.drop_duplicates("symbol").sort_values("symbol").reset_index(drop=True)
    return universe


df_universe = load_symbol_universe()
display(df_universe.head(20))
print(f"Universe symbols: {len(df_universe):,}")


,symbol,asset_name,category
0,AAPL,Apple,MegaCap_Tech
1,ABBV,AbbVie,Healthcare
2,ABNB,Airbnb,Platform
3,ADBE,Adobe,Software
4,AMAT,Applied Materials,Semiconductors
5,AMD,AMD,Semiconductors
6,AMZN,Amazon,MegaCap_Tech
7,ARKK,ARK Innovation ETF,Theme_ETF
8,ASML,ASML,Semiconductors
9,AVGO,Broadcom,Semiconductors


Universe symbols: 100


## 장기 가격 수집

In [4]:
def normalize_price_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rename_map = {}
    for col in out.columns:
        lc = str(col).lower()
        if lc in {'date', 'datetime'}:
            rename_map[col] = 'date'
        elif lc in {'open'}:
            rename_map[col] = 'open'
        elif lc in {'high'}:
            rename_map[col] = 'high'
        elif lc in {'low'}:
            rename_map[col] = 'low'
        elif lc in {'close'}:
            rename_map[col] = 'close'
        elif lc in {'adj_close', 'adj close'}:
            rename_map[col] = 'adj_close'
        elif lc in {'volume'}:
            rename_map[col] = 'volume'
    out = out.rename(columns=rename_map)
    return out


def download_long_history(symbol: str) -> pd.DataFrame:
    data = yf.download(symbol, period=LONG_LOOKBACK_PERIOD, auto_adjust=False, progress=False, group_by='column', threads=False)
    if data is None or len(data) == 0:
        return pd.DataFrame()
    out = data.reset_index()
    if isinstance(out.columns, pd.MultiIndex):
        flat_cols = []
        for col in out.columns:
            parts = [str(x).strip() for x in col if str(x).strip()]
            if len(parts) >= 2 and parts[0].lower() == symbol.lower():
                flat_cols.append(parts[1])
            elif len(parts) >= 2 and parts[-1].lower() == symbol.lower():
                flat_cols.append(parts[0])
            else:
                flat_cols.append(parts[0] if parts else '')
        out.columns = flat_cols
    out.columns = [str(c).strip().lower().replace(' ', '_') for c in out.columns]
    out = normalize_price_columns(out)
    if 'date' not in out.columns:
        if 'index' in out.columns:
            out = out.rename(columns={'index': 'date'})
        else:
            raise ValueError(f'missing date column after download: {list(out.columns)}')
    if 'close' not in out.columns:
        raise ValueError(f'missing close column after download: {list(out.columns)}')
    keep_cols = [c for c in ['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume'] if c in out.columns]
    out = out[keep_cols].copy()
    out['date'] = pd.to_datetime(out['date'], utc=True, errors='coerce')
    out = out.dropna(subset=['date', 'close']).sort_values('date').reset_index(drop=True)
    out = out[out['date'] >= HISTORY_START_CUTOFF].reset_index(drop=True)
    return out


frames = []
failures = []
total = len(df_universe)

for idx, row in enumerate(df_universe.itertuples(index=False), start=1):
    symbol = row.symbol
    print(f"[Long price] {idx}/{total} | {symbol}", flush=True)
    try:
        hist = download_long_history(symbol)
        if hist.empty:
            failures.append({'symbol': symbol, 'reason': 'empty'})
            continue
        hist['symbol'] = symbol
        hist['asset_name'] = row.asset_name
        hist['category'] = row.category
        frames.append(hist)
    except Exception as exc:
        failures.append({'symbol': symbol, 'reason': str(exc)[:300]})

df_long = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
df_failures = pd.DataFrame(failures)
print(f"Downloaded symbols: {df_long['symbol'].nunique() if len(df_long) else 0}")
print(f"Failed symbols: {len(df_failures)}")
if len(df_failures):
    print('Sample failures:')
    display(df_failures.head(20))


[Long price] 1/100 | AAPL
[Long price] 2/100 | ABBV
[Long price] 3/100 | ABNB
[Long price] 4/100 | ADBE
[Long price] 5/100 | AMAT
[Long price] 6/100 | AMD
[Long price] 7/100 | AMZN
[Long price] 8/100 | ARKK
[Long price] 9/100 | ASML
[Long price] 10/100 | AVGO
[Long price] 11/100 | BA
[Long price] 12/100 | BAC
[Long price] 13/100 | BLK
[Long price] 14/100 | BRK-B
[Long price] 15/100 | CAT
[Long price] 16/100 | COP
[Long price] 17/100 | COST
[Long price] 18/100 | CRM
[Long price] 19/100 | CRWD
[Long price] 20/100 | CSCO
[Long price] 21/100 | CVX
[Long price] 22/100 | DE
[Long price] 23/100 | DHR
[Long price] 24/100 | DIA
[Long price] 25/100 | DIS
[Long price] 26/100 | FDX
[Long price] 27/100 | GE
[Long price] 28/100 | GLD
[Long price] 29/100 | GOOGL
[Long price] 30/100 | GS
[Long price] 31/100 | HD
[Long price] 32/100 | HON
[Long price] 33/100 | HYG
[Long price] 34/100 | IBM
[Long price] 35/100 | IEF
[Long price] 36/100 | IGV
[Long price] 37/100 | INTC
[Long price] 38/100 | INTU
[Long pr

## 저장 및 요약

In [5]:
if df_long.empty:
    print("No long price rows collected.")
else:
    df_long = df_long.sort_values(["symbol", "date"]).drop_duplicates(subset=["symbol", "date"], keep="last").reset_index(drop=True)
    LONG_PRICE_MASTER_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_long.to_csv(LONG_PRICE_MASTER_PATH, index=False, encoding="utf-8-sig")
    display(df_long.head())
    print(f"Saved: {LONG_PRICE_MASTER_PATH}")
    print(f"Rows: {len(df_long):,}")
    print(f"Symbols: {df_long['symbol'].nunique():,}")
    coverage = df_long.groupby("symbol", as_index=False).agg(
        rows=("date", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    ).sort_values("rows", ascending=False)
    coverage["history_years"] = ((coverage["last_date"] - coverage["first_date"]).dt.days / 365.25).round(1)
    display(coverage.head(20))


,date,open,high,low,close,adj_close,volume,symbol,asset_name,category
0,1996-03-27 00:00:00+00:00,0.207589,0.225446,0.205357,0.225446,0.189141,429296000,AAPL,Apple,MegaCap_Tech
1,1996-03-28 00:00:00+00:00,0.220982,0.228795,0.215402,0.215960,0.181183,295892800,AAPL,Apple,MegaCap_Tech
2,1996-03-29 00:00:00+00:00,0.216518,0.220982,0.212054,0.219308,0.183992,166521600,AAPL,Apple,MegaCap_Tech
3,1996-04-01 00:00:00+00:00,0.224330,0.231027,0.218890,0.227679,0.191015,158636800,AAPL,Apple,MegaCap_Tech
4,1996-04-02 00:00:00+00:00,0.228795,0.228795,0.222098,0.223214,0.187269,101438400,AAPL,Apple,MegaCap_Tech


Saved: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\long_price_master_history.csv
Rows: 641,522
Symbols: 100


,symbol,rows,first_date,last_date,history_years
3,ADBE,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
10,BA,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
5,AMD,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
4,AMAT,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
31,HON,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
30,HD,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
33,IBM,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
8,ASML,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
14,CAT,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
25,FDX,7549,1996-03-27 00:00:00+00:00,2026-03-27 00:00:00+00:00,30.0
